# Direct Demand Model for Corridor Ridership — Full Analysis
### Economy & 1st Class · static vs lagged-demand · baseline vs multicollinearity-refined

Self-contained notebook (supersedes `TransitRidershipDemandModel.ipynb`). Run top to bottom;
every printed interpretation is computed from the fitted models, not hardcoded.

| Part | Sections |
|---|---|
| **Setup & exploration** | 1 Setup · 2 Load · 3 Prep · 4 Correlations · 5 Seasonality by year |
| **A. Baseline (full) specification** | 6-7 Static models · 8 VIF · 9-10 Dynamic (lagged) models · 11 Short/long-run · 12 Autocorrelation · 13 Static vs dynamic |
| **B. Addressing multicollinearity** | 14 Remedy variants · 15 Choosing the refined specification |
| **C. Refined specification** | 16 VIF · 17-18 Static & dynamic models · 19 Short/long-run · 20 Autocorrelation |
| **D. Model selection & results** | 21 Master comparison · 22 Final model per class · 23 Final elasticities · 24 Conclusions |

**Methodological note.** AIC/BIC are only comparable between models fitted to the *same*
observations. Static models use 36 quarters, dynamic models 35 (the first quarter has no lag),
so every static-vs-dynamic comparison below refits the static model on the common 35-quarter sample.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 40)
ALPHA = 0.05
SRC_PATH = "TransitRidershipData.xlsx"  # run from the folder containing the data file


## 2. Load & Inspect Data
Confirm the raw file loaded correctly before trusting anything downstream.


In [ ]:
raw = pd.read_excel(SRC_PATH)
print(f"Loaded {raw.shape[0]} rows x {raw.shape[1]} columns")
display(raw.head())
print("Missing values:", int(raw.isnull().sum().sum()))
display(raw.describe().T)


## 3. Data Preparation
- Short column names; log-transform ridership and continuous price/economic regressors
  (**log-log** form → coefficients are elasticities). `Delays` stays in levels (**semi-elasticity**).
- Quarter dummies `Q_2, Q_3, Q_4` (Q1 = reference) capture seasonality.
- One-quarter lag of each class's own log-ridership for the dynamic models.
- Extra variables used only by the collinearity remedies in Part B: a linear `trend` and the
  relative fare `ln_FareRatio = ln(Fare_1st) - ln(Fare_econ)`.


In [ ]:
df = raw.sort_values(["Year", "Quarter"]).reset_index(drop=True)
df["t"] = range(1, len(df) + 1)
df["trend"] = df["t"]

df = df.rename(columns={
    "Total Number of Passengers: Economy Class": "Q_econ",
    "Total Number of Passengers: 1st Class": "Q_1st",
    "Avg Fare Per KM of Economyclass": "Fare_econ",
    "Avg Fare Per KM of 1st class": "Fare_1st",
    "On Time Performances: Number of Times Schedule Delays happened": "Delays",
    "Total Employment in the Region along the Corridor": "Employment",
    "Avg Gas Price along the Corridor": "GasPrice",
    "Avg Air Fare Per KM along the Corridor": "AirFare",
})
for col in ["Q_econ", "Q_1st", "Fare_econ", "Fare_1st", "Employment", "GasPrice", "AirFare"]:
    df[f"ln_{col}"] = np.log(df[col])
df["ln_FareRatio"] = df["ln_Fare_1st"] - df["ln_Fare_econ"]

df["Quarter"] = df["Quarter"].astype(int)
df = pd.get_dummies(df, columns=["Quarter"], prefix="Q", drop_first=True)
for c in ["Q_2", "Q_3", "Q_4"]:
    df[c] = df[c].astype(int)

df["ln_Q_econ_lag1"] = df["ln_Q_econ"].shift(1)
df["ln_Q_1st_lag1"] = df["ln_Q_1st"].shift(1)
df_dyn = df.dropna(subset=["ln_Q_econ_lag1", "ln_Q_1st_lag1"]).copy()

SEAS = ["Q_2", "Q_3", "Q_4"]
BASE_COLS = ["ln_Fare_econ", "ln_Fare_1st", "ln_AirFare", "ln_GasPrice", "ln_Employment", "Delays"]

CLASSES = {
    "Economy":   dict(dep="ln_Q_econ", lag="ln_Q_econ_lag1", own="ln_Fare_econ", cross="ln_Fare_1st"),
    "1st Class": dict(dep="ln_Q_1st",  lag="ln_Q_1st_lag1",  own="ln_Fare_1st",  cross="ln_Fare_econ"),
}

print(f"Static sample: {len(df)} quarters | common/dynamic sample: {len(df_dyn)} quarters")
display(df[["Year", "t", "Q_2", "Q_3", "Q_4", "ln_Q_econ", "ln_Q_econ_lag1", "ln_Q_1st", "ln_Q_1st_lag1"]].head(6))


In [ ]:
# ---- shared helpers used by every later section ----
def fit(cls, cols, data, lagged=False):
    c = CLASSES[cls]
    rhs = ([c["lag"]] if lagged else []) + cols + SEAS
    return smf.ols(f"{c['dep']} ~ " + " + ".join(rhs), data=data).fit()

def meta_for(cls):
    c = CLASSES[cls]
    m = {
        "ln_AirFare":    ("elasticity", "Competing air fare"),
        "ln_GasPrice":   ("elasticity", "Gas price (driving-cost proxy)"),
        "ln_Employment": ("elasticity", "Regional employment (economic activity)"),
        "ln_FareRatio":  ("elasticity", "Relative fare, 1st Class / Economy"),
        "trend":         ("semi-elasticity", "Linear time trend (per quarter)"),
        "Delays":        ("semi-elasticity", "Schedule-delay count (service quality)"),
        "Q_2": ("dummy", "Q2 vs Q1"), "Q_3": ("dummy", "Q3 vs Q1"), "Q_4": ("dummy", "Q4 vs Q1"),
    }
    m[c["own"]] = ("elasticity", f"{cls} OWN fare")
    m[c["cross"]] = ("elasticity", "CROSS fare (other class)")
    return m

def interpret_model(model, cls, alpha=ALPHA):
    c = CLASSES[cls]; meta = meta_for(cls)
    print(f"R2={model.rsquared:.3f}  Adj.R2={model.rsquared_adj:.3f}  N={int(model.nobs)}  "
          f"F p-value={model.f_pvalue:.2e}  AIC={model.aic:.2f}  BIC={model.bic:.2f}")
    print("-" * 105)
    for var in model.params.index:
        if var == "Intercept":
            continue
        b, p = model.params[var], model.pvalues[var]
        sig = "significant" if p < alpha else "not significant"
        star = "*" if p < alpha else " "
        if var == c["lag"]:
            print(f"{star} {var:15s} gamma={b:+.4f} p={p:.3f} ({sig}) -> speed of adjustment (habit persistence)")
            continue
        kind, label = meta.get(var, ("elasticity", var))
        if kind == "elasticity":
            print(f"{star} {var:15s} b={b:+.4f} p={p:.3f} ({sig}) -> {label}: +1% -> {b:+.3f}% ridership")
        elif kind == "semi-elasticity":
            print(f"{star} {var:15s} b={b:+.5f} p={p:.3f} ({sig}) -> {label}: +1 unit -> {b*100:+.3f}% ridership")
        else:
            print(f"{star} {var:15s} b={b:+.4f} p={p:.3f} ({sig}) -> {label}: {(np.exp(b)-1)*100:+.1f}% ridership")
    print("-" * 105)
    print(f"(* = significant at {int(alpha*100)}%)")

def vif_table(data, cols):
    X = data[cols + SEAS].copy(); X.insert(0, "const", 1.0)
    out = pd.DataFrame({"variable": X.columns,
                        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
    return out[out["variable"] != "const"].reset_index(drop=True)

def show_vif(data, cols, title):
    v = vif_table(data, cols)
    print(title); display(v.round(2))
    bad = v[v["VIF"] > 10]
    print("VIF > 10:", ", ".join(bad["variable"]) if len(bad) else "none")

def sr_lr_table(model, lag_name):
    g = model.params[lag_name]
    rows = [(v, model.params[v], model.params[v] / (1 - g), model.pvalues[v])
            for v in model.params.index if v not in ("Intercept", lag_name)]
    out = pd.DataFrame(rows, columns=["variable", "short_run", "long_run", "p_value"])
    out.attrs["gamma"] = g
    return out

def durbins_h(model, lag_name):
    n = int(model.nobs); dw = durbin_watson(model.resid)
    denom = 1 - n * model.bse[lag_name] ** 2
    return dw, (1 - dw / 2) * np.sqrt(n / denom) if denom > 0 else None

def report_autocorr(static_m, dyn_m, lag_name, label):
    dw_s = durbin_watson(static_m.resid); dw_d, h = durbins_h(dyn_m, lag_name)
    print(f"{label}: DW static = {dw_s:.3f} | DW dynamic = {dw_d:.3f} | "
          f"Durbin's h = {'undefined (n*se^2 >= 1, small sample)' if h is None else round(h, 3)}")


## 4. Exploratory Correlations
Pairwise correlations flag candidate multicollinearity *before* modeling (VIF in Sections 8/16 confirms it).


In [ ]:
corr_vars = ["ln_Q_econ", "ln_Q_1st", "ln_Fare_econ", "ln_Fare_1st",
             "ln_AirFare", "ln_GasPrice", "ln_Employment", "Delays"]
corr = df[corr_vars].corr().round(2)
display(corr)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(corr_vars))); ax.set_xticklabels(corr_vars, rotation=45, ha="right")
ax.set_yticks(range(len(corr_vars))); ax.set_yticklabels(corr_vars)
for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="Pearson r"); ax.set_title("Correlation matrix"); plt.tight_layout(); plt.show()
print("Pairs with |r| > 0.7:")
for i in range(len(corr_vars)):
    for j in range(i + 1, len(corr_vars)):
        if abs(corr.values[i, j]) > 0.7:
            print(f"  {corr_vars[i]} <-> {corr_vars[j]}: {corr.values[i, j]:.2f}")


## 5. Seasonality by Year
The quarter dummies assume one seasonal pattern *pooled across all 9 years*. This section checks that
assumption: each line is one year's ridership as a share of that year's average, so a stable seasonal
shape shows as lines that sit on top of each other.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=False)
season_tables = {}
for ax, (cls, col) in zip(axes, [("Economy", "Q_econ"), ("1st Class", "Q_1st")]):
    q = raw.rename(columns={"Total Number of Passengers: Economy Class": "Q_econ",
                            "Total Number of Passengers: 1st Class": "Q_1st"})
    piv = q.pivot(index="Year", columns="Quarter", values=col)
    idx = piv.div(piv.mean(axis=1), axis=0)          # quarter / annual mean
    season_tables[cls] = idx
    for yr, row in idx.iterrows():
        ax.plot(idx.columns, row.values, marker="o", alpha=0.7, label=f"Year {yr}")
    ax.axhline(1, color="grey", lw=0.8, ls="--")
    ax.set_title(f"{cls}: quarterly ridership / annual mean"); ax.set_xticks([1, 2, 3, 4]); ax.set_xlabel("Quarter")
axes[0].legend(ncol=3, fontsize=7); plt.tight_layout(); plt.show()

for cls, idx in season_tables.items():
    print(f"{cls}: mean seasonal index by quarter and spread across years (std)")
    display(pd.DataFrame({"mean": idx.mean(), "std_across_years": idx.std()}).T.round(3))
print("Small std relative to the gaps between quarters supports the pooled (constant-seasonality) assumption.")


# PART A — Baseline (full) specification
`ln(Q) = b0 + b1 ln(Fare_econ) + b2 ln(Fare_1st) + b3 ln(AirFare) + b4 ln(GasPrice) + b5 ln(Employment) + b6 Delays + Q2 + Q3 + Q4`

## 6. Baseline Static Model — Economy


In [ ]:
m_econ_static = fit("Economy", BASE_COLS, df)
print(m_econ_static.summary())


In [ ]:
print("INTERPRETATION - Economy, baseline static")
interpret_model(m_econ_static, "Economy")


## 7. Baseline Static Model — 1st Class

In [ ]:
m_1st_static = fit("1st Class", BASE_COLS, df)
print(m_1st_static.summary())


In [ ]:
print("INTERPRETATION - 1st Class, baseline static")
interpret_model(m_1st_static, "1st Class")


## 8. Multicollinearity Check (VIF) — Baseline
VIF > 10 is the usual warning threshold: standard errors inflate and otherwise sensible coefficients turn insignificant or change sign.


In [ ]:
show_vif(df, BASE_COLS, "Baseline regressor set")


## 9. Baseline Dynamic Model (Lagged Demand) — Economy
`ln(Q_t) = b0 + gamma*ln(Q_t-1) + ...` — **gamma** is the habit-persistence / adjustment-speed parameter;
long-run elasticity = short-run coefficient / (1 - gamma).


In [ ]:
m_econ_dyn = fit("Economy", BASE_COLS, df_dyn, lagged=True)
print(m_econ_dyn.summary())


In [ ]:
print("INTERPRETATION - Economy, baseline dynamic")
interpret_model(m_econ_dyn, "Economy")
g, p = m_econ_dyn.params["ln_Q_econ_lag1"], m_econ_dyn.pvalues["ln_Q_econ_lag1"]
print(f"\n=> gamma = {g:.3f}, p = {p:.3f}: lagged demand is {'' if p < ALPHA else 'NOT '}statistically significant at 5%.")


## 10. Baseline Dynamic Model (Lagged Demand) — 1st Class

In [ ]:
m_1st_dyn = fit("1st Class", BASE_COLS, df_dyn, lagged=True)
print(m_1st_dyn.summary())


In [ ]:
print("INTERPRETATION - 1st Class, baseline dynamic")
interpret_model(m_1st_dyn, "1st Class")
g, p = m_1st_dyn.params["ln_Q_1st_lag1"], m_1st_dyn.pvalues["ln_Q_1st_lag1"]
print(f"\n=> gamma = {g:.3f}, p = {p:.3f}: lagged demand is {'' if p < ALPHA else 'NOT '}statistically significant at 5%"
      f" ({'' if p < 0.10 else 'not '}at 10%).")


## 11. Baseline Short-Run vs Long-Run Elasticities

In [ ]:
for cls, m, lag in [("Economy", m_econ_dyn, "ln_Q_econ_lag1"), ("1st Class", m_1st_dyn, "ln_Q_1st_lag1")]:
    t = sr_lr_table(m, lag)
    print(f"{cls}: gamma = {t.attrs['gamma']:.4f}")
    display(t.round(4))


## 12. Baseline Autocorrelation Diagnostics
Durbin-Watson (DW) ≈ 2 means no first-order residual autocorrelation; well below 2 means positive autocorrelation
(standard errors too small). DW is biased toward 2 with a lagged dependent variable, so Durbin's h is the formal
test there — but it is undefined when `n*se(gamma)^2 >= 1`, common in small samples.


In [ ]:
report_autocorr(m_econ_static, m_econ_dyn, "ln_Q_econ_lag1", "Economy  ")
report_autocorr(m_1st_static, m_1st_dyn, "ln_Q_1st_lag1", "1st Class")


## 13. Baseline: Static vs Dynamic on a Common Sample
Both models refit on the same 35 quarters so AIC/BIC are comparable (lower = better).


In [ ]:
def compare_static_dynamic(cls, cols):
    c = CLASSES[cls]
    s = fit(cls, cols, df_dyn, lagged=False)
    d = fit(cls, cols, df_dyn, lagged=True)
    return pd.DataFrame({
        "model": ["static", "dynamic"],
        "N": [int(s.nobs), int(d.nobs)],
        "adj_R2": [s.rsquared_adj, d.rsquared_adj],
        "AIC": [s.aic, d.aic], "BIC": [s.bic, d.bic],
        "DW": [durbin_watson(s.resid), durbin_watson(d.resid)],
        "gamma": [np.nan, d.params[c["lag"]]], "gamma_p": [np.nan, d.pvalues[c["lag"]]],
    }).round(4)

for cls in CLASSES:
    print(f"{cls} - baseline specification"); display(compare_static_dynamic(cls, BASE_COLS))


# PART B — Addressing multicollinearity

## 14. Remedy Variants
Section 4/8 showed `ln_Employment` (and `ln_Fare_1st`) collinear with other regressors. Six specifications are compared
on the full 36-quarter sample (all static, so AIC/BIC are comparable):

| Variant | Change |
|---|---|
| A | Baseline (all six regressors) |
| B | Drop `GasPrice` |
| C | Replace `Employment` with a linear time trend |
| D | Replace both fares with the relative fare ln(Fare_1st/Fare_econ) |
| E | Drop `GasPrice` **and** use the relative fare |
| F | Drop `GasPrice` **and** replace `Employment` with the trend |


In [ ]:
VARIANTS = {
    "A. Baseline (full)":              BASE_COLS,
    "B. Drop GasPrice":                ["ln_Fare_econ", "ln_Fare_1st", "ln_AirFare", "ln_Employment", "Delays"],
    "C. Employment -> trend":          ["ln_Fare_econ", "ln_Fare_1st", "ln_AirFare", "ln_GasPrice", "trend", "Delays"],
    "D. Fare ratio (1st/econ)":        ["ln_FareRatio", "ln_AirFare", "ln_GasPrice", "ln_Employment", "Delays"],
    "E. Drop Gas + fare ratio":        ["ln_FareRatio", "ln_AirFare", "ln_Employment", "Delays"],
    "F. Drop Gas, Employment -> trend": ["ln_Fare_econ", "ln_Fare_1st", "ln_AirFare", "trend", "Delays"],
}
COEF_SHOW = ["ln_Fare_econ", "ln_Fare_1st", "ln_FareRatio", "ln_AirFare", "ln_GasPrice", "ln_Employment", "trend", "Delays"]

variant_fit, fit_rows, coef_rows = {}, [], []
for cls in CLASSES:
    for name, cols in VARIANTS.items():
        m = fit(cls, cols, df)
        v = vif_table(df, cols).set_index("variable")["VIF"]
        variant_fit[(cls, name)] = m
        fit_rows.append(dict(cls=cls, variant=name, adj_R2=m.rsquared_adj, AIC=m.aic, BIC=m.bic,
                             DW=durbin_watson(m.resid), max_VIF=v.max(), worst_var=v.idxmax()))
        r = dict(cls=cls, variant=name)
        for k in COEF_SHOW:
            r[k] = f"{m.params[k]:+.3f} (p={m.pvalues[k]:.3f})" if k in m.params.index else "-"
        coef_rows.append(r)
fit_df, coef_df = pd.DataFrame(fit_rows), pd.DataFrame(coef_rows)

for cls in CLASSES:
    print("=" * 110); print(f"{cls} - fit, diagnostics and multicollinearity by variant"); print("=" * 110)
    display(fit_df[fit_df["cls"] == cls].drop(columns="cls").round(3).reset_index(drop=True))
    print(f"{cls} - key coefficients by variant (coefficient, p-value)")
    display(coef_df[coef_df["cls"] == cls].drop(columns="cls").reset_index(drop=True))


## 15. Choosing the Refined Specification
Criteria: (i) no VIF above 10, (ii) best (lowest) BIC among variants — BIC penalises extra parameters most, which matters with
only 36 observations — and (iii) no loss of fit versus the baseline. The ranking below is computed; the choice is then set explicitly.


In [ ]:
for cls in CLASSES:
    sub = fit_df[fit_df["cls"] == cls].copy()
    sub["VIF_ok"] = sub["max_VIF"] < 10
    sub["BIC_rank"] = sub["BIC"].rank().astype(int)
    print(f"{cls}: variants ranked by BIC (rank 1 = best)")
    display(sub.sort_values("BIC")[["variant", "BIC", "AIC", "adj_R2", "max_VIF", "VIF_ok", "BIC_rank"]].round(3).reset_index(drop=True))

REFINED_NAME = "B. Drop GasPrice"      # <- analyst's choice; change to test another variant
REFINED_COLS = VARIANTS[REFINED_NAME]
print("Refined specification used from here on:", REFINED_NAME)
print("Regressors:", REFINED_COLS)


# PART C — Refined specification
Same models as Part A with the chosen remedy applied.

## 16. VIF — Refined


In [ ]:
show_vif(df, REFINED_COLS, f"Refined regressor set ({REFINED_NAME})")


## 17. Refined Static Models

In [ ]:
r_econ_static = fit("Economy", REFINED_COLS, df)
print(r_econ_static.summary())


In [ ]:
print("INTERPRETATION - Economy, refined static")
interpret_model(r_econ_static, "Economy")


In [ ]:
r_1st_static = fit("1st Class", REFINED_COLS, df)
print(r_1st_static.summary())


In [ ]:
print("INTERPRETATION - 1st Class, refined static")
interpret_model(r_1st_static, "1st Class")


## 18. Refined Dynamic Models (Lagged Demand)

In [ ]:
r_econ_dyn = fit("Economy", REFINED_COLS, df_dyn, lagged=True)
print(r_econ_dyn.summary())


In [ ]:
print("INTERPRETATION - Economy, refined dynamic")
interpret_model(r_econ_dyn, "Economy")
g, p = r_econ_dyn.params["ln_Q_econ_lag1"], r_econ_dyn.pvalues["ln_Q_econ_lag1"]
print(f"\n=> gamma = {g:.3f}, p = {p:.3f}: lagged demand is {'' if p < ALPHA else 'NOT '}significant at 5%.")


In [ ]:
r_1st_dyn = fit("1st Class", REFINED_COLS, df_dyn, lagged=True)
print(r_1st_dyn.summary())


In [ ]:
print("INTERPRETATION - 1st Class, refined dynamic")
interpret_model(r_1st_dyn, "1st Class")
g, p = r_1st_dyn.params["ln_Q_1st_lag1"], r_1st_dyn.pvalues["ln_Q_1st_lag1"]
print(f"\n=> gamma = {g:.3f}, p = {p:.3f}: lagged demand is {'' if p < ALPHA else 'NOT '}significant at 5% "
      f"({'' if p < 0.10 else 'not '}at 10%; one-sided p for gamma>0 is about {p/2:.3f}).")


## 19. Refined Short-Run vs Long-Run Elasticities

In [ ]:
for cls, m, lag in [("Economy", r_econ_dyn, "ln_Q_econ_lag1"), ("1st Class", r_1st_dyn, "ln_Q_1st_lag1")]:
    t = sr_lr_table(m, lag)
    print(f"{cls}: gamma = {t.attrs['gamma']:.4f}")
    display(t.round(4))


## 20. Refined Autocorrelation Diagnostics

In [ ]:
r_econ_static35 = fit("Economy", REFINED_COLS, df_dyn)
r_1st_static35 = fit("1st Class", REFINED_COLS, df_dyn)
report_autocorr(r_econ_static35, r_econ_dyn, "ln_Q_econ_lag1", "Economy  ")
report_autocorr(r_1st_static35, r_1st_dyn, "ln_Q_1st_lag1", "1st Class")
print("\n(static DW shown on the common 35-quarter sample)")


# PART D — Model selection and final results

## 21. Master Comparison — All Four Specifications per Class
Baseline/refined × static/dynamic, all on the common 35-quarter sample.


In [ ]:
master_rows = []
for cls in CLASSES:
    lag = CLASSES[cls]["lag"]
    for spec, cols in [("baseline", BASE_COLS), ("refined", REFINED_COLS)]:
        for form, lagged in [("static", False), ("dynamic", True)]:
            m = fit(cls, cols, df_dyn, lagged=lagged)
            master_rows.append(dict(
                cls=cls, spec=spec, form=form, k=int(m.df_model) + 1,
                adj_R2=m.rsquared_adj, AIC=m.aic, BIC=m.bic, DW=durbin_watson(m.resid),
                gamma=m.params[lag] if lagged else np.nan, gamma_p=m.pvalues[lag] if lagged else np.nan,
                max_VIF=vif_table(df_dyn, cols + ([lag] if lagged else [])).VIF.max()))
master = pd.DataFrame(master_rows)
for cls in CLASSES:
    print(f"{cls}"); display(master[master["cls"] == cls].drop(columns="cls").round(3).reset_index(drop=True))


## 22. Final Model per Class
Decision aids computed from the refined specification: the lag coefficient's significance, the static→dynamic change in AIC and BIC
(negative = dynamic better; |ΔAIC| < 2 is not meaningful evidence either way), and residual autocorrelation.


In [ ]:
aid = []
for cls in CLASSES:
    s = master[(master.cls == cls) & (master.spec == "refined") & (master.form == "static")].iloc[0]
    d = master[(master.cls == cls) & (master.spec == "refined") & (master.form == "dynamic")].iloc[0]
    aid.append(dict(cls=cls, gamma=d.gamma, gamma_p=d.gamma_p,
                    dAIC=d.AIC - s.AIC, dBIC=d.BIC - s.BIC, DW_static=s.DW, DW_dynamic=d.DW,
                    dAdjR2=d.adj_R2 - s.adj_R2))
display(pd.DataFrame(aid).round(3))

# Analyst judgement, informed by the table above (edit and re-run to test alternatives):
FINAL = {"Economy": "static", "1st Class": "dynamic"}
FINAL_MODELS = {
    "Economy":   r_econ_static if FINAL["Economy"] == "static" else r_econ_dyn,
    "1st Class": r_1st_static if FINAL["1st Class"] == "static" else r_1st_dyn,
}
print("Selected:", {k: f"refined {v}" for k, v in FINAL.items()})


**Reading the evidence.** Economy: the lag is small and insignificant, AIC/BIC prefer the static model → static. 1st Class: the lag has the
expected positive sign, ΔAIC favours the dynamic model, BIC is roughly tied, adjusted R² is higher, and — importantly — the dynamic model removes the
positive residual autocorrelation visible in the static model (DW), which otherwise makes the static standard errors unreliable. The lag coefficient itself
is significant only at roughly the 10-12% level, so treat the choice of the dynamic form for 1st Class as a *judgement call supported by AIC and the
autocorrelation diagnostic*, not a decisive statistical result.


## 23. Final Elasticities
Elasticities of the selected model per class. Log-log coefficients are elasticities; `Delays` is a semi-elasticity;
quarter dummies are converted to % differences vs Q1. For a dynamic model the long-run value is short-run / (1 - gamma).


In [ ]:
def final_table(cls):
    m = FINAL_MODELS[cls]; c = CLASSES[cls]; meta = meta_for(cls)
    g = m.params.get(c["lag"], 0.0)
    rows = []
    for v in m.params.index:
        if v in ("Intercept", c["lag"]):
            continue
        kind, label = meta.get(v, ("elasticity", v))
        b = m.params[v]
        eff = (np.exp(b) - 1) * 100 if kind == "dummy" else (b * 100 if kind == "semi-elasticity" else b)
        unit = "% vs Q1" if kind == "dummy" else ("% per unit" if kind == "semi-elasticity" else "elasticity")
        rows.append(dict(variable=v, meaning=label, short_run=eff, long_run=eff / (1 - g) if (c["lag"] in m.params.index and kind != "dummy") else np.nan,
                         unit=unit, p_value=m.pvalues[v], significant="yes" if m.pvalues[v] < ALPHA else ("10%" if m.pvalues[v] < 0.10 else "no")))
    return pd.DataFrame(rows), g

for cls in CLASSES:
    t, g = final_table(cls)
    kind = "dynamic (gamma = %.3f)" % g if FINAL[cls] == "dynamic" else "static"
    print(f"{cls} - final model: refined {kind}")
    display(t.round(4))


## 24. Conclusions
- **Multicollinearity.** Dropping `GasPrice` brings every VIF below 10 with no loss of fit (better BIC in both classes) and is the best of the
  six remedies; the relative-fare variants fit clearly worse, especially for 1st Class.
- **What the remedy does *not* fix.** Own-fare elasticities remain wrong-signed (positive) in both classes, and for 1st Class the refined static
  model makes this significant. That is unlikely to be collinearity; a more probable cause is price *endogeneity* — average fare per km can rise
  because demand is high (yield management), so the regression captures demand pushing fares up rather than fares pushing demand down. Own-fare
  elasticities should therefore not be used for fare-policy conclusions without an instrument or a better price measure.
- **Lagged demand.** Economy: no evidence (γ small, insignificant; static preferred). 1st Class: weak-to-moderate evidence (γ ≈ 0.28, p ≈ 0.11,
  ΔAIC favours dynamic, autocorrelation removed) — reported as a judgement, not a firm finding, given only 35 observations.
- **Reliable results.** Economy demand is driven by regional employment (elasticity ≈ 1) and competing air fare (≈ 0.2); both classes show strong,
  stable seasonality (Section 5), with 1st Class swinging roughly twice as much as Economy.
- **Caveats.** Small sample; seasonality assumed constant across years; no formal Durbin's h (undefined here); possible omitted variables.
